In [12]:
import pandas as pd
from pathlib import Path

In [13]:
# Folder containing feature datasets
DATA_DIR = Path("features")

geometry_file = DATA_DIR / "geometry_features.csv"
zoning_file = DATA_DIR / "functional_zoning_features.csv"
syntax_file = DATA_DIR / "spatial_syntax_features.csv"
plan_file = DATA_DIR / "plan_level_features.csv"

output_file = Path("floorplan_features.csv")

In [14]:
geometry_df = pd.read_csv(geometry_file)
zoning_df = pd.read_csv(zoning_file)
syntax_df = pd.read_csv(syntax_file)
plan_df = pd.read_csv(plan_file)

print("Geometry dataset:", geometry_df.shape)
print("Functional zoning dataset:", zoning_df.shape)
print("Spatial syntax dataset:", syntax_df.shape)
print("Plan-level dataset:", plan_df.shape)

Geometry dataset: (4992, 7)
Functional zoning dataset: (4992, 4)
Spatial syntax dataset: (4992, 5)
Plan-level dataset: (4998, 12)


In [15]:
datasets = {
    "geometry": geometry_df,
    "zoning": zoning_df,
    "syntax": syntax_df,
    "plan": plan_df
}

for name, df in datasets.items():
    if "plan_id" not in df.columns:
        print(f"ERROR: {name} dataset missing plan_id")
    else:
        print(f"{name} dataset OK")

geometry dataset OK
zoning dataset OK
syntax dataset OK
plan dataset OK


In [16]:
merged_df = geometry_df.merge(zoning_df, on="plan_id", how="inner")
print("After merging geometry + zoning:", merged_df.shape)

merged_df = merged_df.merge(syntax_df, on="plan_id", how="inner")
print("After adding spatial syntax:", merged_df.shape)

merged_df = merged_df.merge(plan_df, on="plan_id", how="inner")
print("After adding plan-level:", merged_df.shape)

After merging geometry + zoning: (4992, 10)
After adding spatial syntax: (4992, 14)
After adding plan-level: (4990, 25)


In [17]:
merged_df = merged_df.loc[:, ~merged_df.columns.duplicated()]

In [18]:
merged_df = merged_df.sort_values("plan_id")

In [19]:
merged_df.head()

,plan_id,num_rooms_x,avg_room_area_x,std_room_area_x,avg_aspect_ratio,avg_compactness,avg_rectangularity,public_private_separation,service_area_ratio,bathroom_adjacency,...,gross_area,usable_area,efficiency,corridor_ratio,avg_room_area_y,std_room_area_y,rooms_with_window_ratio,external_wall_ratio,cross_ventilation_ratio,window_wall_ratio
0,1,14,7.487408,6.326511,1.789442,0.649941,0.975016,1.0,0.142857,0.0,...,104.823713,101.739174,0.970574,0.029426,7.487408,6.096378,0.500000,0.432432,0.0,0.020532
1,2,7,6.913880,6.906330,2.267390,0.669636,0.997796,1.0,0.142857,0.0,...,48.397162,43.835059,0.905736,0.094264,6.913880,6.394019,0.142857,0.545455,0.0,0.018665
2,3,5,8.834605,9.940902,1.923613,0.673939,0.960626,1.0,0.200000,0.0,...,44.173027,40.201286,0.910087,0.089913,8.834605,8.891413,0.200000,0.769231,0.2,0.019516
3,4,5,12.932369,9.314597,1.916750,0.630721,0.937192,1.0,0.200000,0.0,...,64.661846,56.365572,0.871698,0.128302,12.932369,8.331229,0.600000,0.608696,0.0,0.022149
4,6,7,9.376730,5.578743,1.693088,0.689853,0.947068,1.0,0.285714,0.0,...,65.637108,65.637108,1.000000,0.000000,9.376730,5.164912,0.571429,0.588235,0.0,0.019049


In [20]:
print("Total columns:", len(merged_df.columns))

Total columns: 25


In [21]:
# Remove duplicated feature columns created during merging

duplicate_cols = [
    "num_rooms_y",
    "avg_room_area_y",
    "std_room_area_y"
]

merged_df = merged_df.drop(columns=duplicate_cols, errors="ignore")

# Rename remaining columns for clarity
merged_df = merged_df.rename(columns={
    "num_rooms_x": "num_rooms",
    "avg_room_area_x": "avg_room_area",
    "std_room_area_x": "std_room_area"
})

print("Final columns:", len(merged_df.columns))
merged_df.head()

Final columns: 22


,plan_id,num_rooms,avg_room_area,std_room_area,avg_aspect_ratio,avg_compactness,avg_rectangularity,public_private_separation,service_area_ratio,bathroom_adjacency,...,mean_depth,integration,gross_area,usable_area,efficiency,corridor_ratio,rooms_with_window_ratio,external_wall_ratio,cross_ventilation_ratio,window_wall_ratio
0,1,14,7.487408,6.326511,1.789442,0.649941,0.975016,1.0,0.142857,0.0,...,1.000000,1.00,104.823713,101.739174,0.970574,0.029426,0.500000,0.432432,0.0,0.020532
1,2,7,6.913880,6.906330,2.267390,0.669636,0.997796,1.0,0.142857,0.0,...,1.333333,0.75,48.397162,43.835059,0.905736,0.094264,0.142857,0.545455,0.0,0.018665
2,3,5,8.834605,9.940902,1.923613,0.673939,0.960626,1.0,0.200000,0.0,...,1.000000,1.00,44.173027,40.201286,0.910087,0.089913,0.200000,0.769231,0.2,0.019516
3,4,5,12.932369,9.314597,1.916750,0.630721,0.937192,1.0,0.200000,0.0,...,1.000000,1.00,64.661846,56.365572,0.871698,0.128302,0.600000,0.608696,0.0,0.022149
4,6,7,9.376730,5.578743,1.693088,0.689853,0.947068,1.0,0.285714,0.0,...,0.000000,0.00,65.637108,65.637108,1.000000,0.000000,0.571429,0.588235,0.0,0.019049


In [22]:
merged_df.to_csv(output_file, index=False)

print("Merged dataset saved as:", output_file)
print("Final dataset shape:", merged_df.shape)

Merged dataset saved as: floorplan_features.csv
Final dataset shape: (4990, 22)
